In [5]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import os
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.linear_model import LogisticRegression

# -----------------------------------------------------------------------------
# 1) Configuration
# -----------------------------------------------------------------------------
# Root directory where resampled data files are stored (no subfolders)
DATA_ROOT     = os.path.join("..", "data", "resampled")
# Directory for saving model outputs (signals)
RESULTS_ROOT  = "results"
# Directory for saving feature importances
FEATURES_ROOT = "features"
# Time intervals to process
INTERVALS     = ["1min", "10min", "1h", "1d"]
# Hyperparameter grid for Logistic Regression
PARAM_GRID    = {"clf__C": [0.1, 1.0, 10.0]}
# Number of splits for time-series cross-validation
N_SPLITS_CV   = 5

# -----------------------------------------------------------------------------
# 2) Load & preprocess data
# -----------------------------------------------------------------------------
def load_and_preprocess(path: str) -> pd.DataFrame:
    # Read parquet file
    df = pd.read_parquet(path)
    # Parse timestamp column to datetime
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    # Sort by coin_id and timestamp to ensure chronological order
    df.sort_values(["coin_id", "timestamp"], inplace=True)
    # Create target: 1 if next close price > current close price, else 0
    df["target_direction"] = (
        df.groupby("coin_id", observed=True)["close"]
          .shift(-1)
          .gt(df["close"])
          .astype(int)
    )
    # Drop rows where target is NaN (end of each coin's series)
    df.dropna(subset=["target_direction"], inplace=True)
    return df

# -----------------------------------------------------------------------------
# 3) Feature engineering
# -----------------------------------------------------------------------------
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # Ensure chronological order before feature computation
    df.sort_values(["coin_id", "timestamp"], inplace=True)
    # Volume ratio: taker buy volume / total quote volume
    df["volume_ratio"] = df["taker_buy_quote_asset_volume"] / df["quote_asset_volume"]
    # Log return: log(close_t) - log(close_{t-1})
    df["log_ret"]      = df.groupby("coin_id")["close"].transform(lambda x: np.log(x).diff())
    # Intraday range percentage: (high - low) / open
    df["range_pct"]    = (df["high"] - df["low"]) / df["open"]
    # Open-close percentage change: (close - open) / open
    df["oc_pct"]       = (df["close"] - df["open"]) / df["open"]
    # 10-period simple moving average
    df["sma_10"]       = df.groupby("coin_id")["close"].transform(lambda x: x.rolling(10).mean())
    # 10-period exponential moving average
    df["ema_10"]       = df.groupby("coin_id")["close"].transform(lambda x: x.ewm(span=10, adjust=False).mean())
    # Rolling mean and volatility for windows [5,10,20,50]
    for w in [5, 10, 20, 50]:
        df[f"ret_ma_{w}"] = df.groupby("coin_id")["log_ret"].transform(lambda x: x.rolling(w).mean())
        df[f"vol_{w}"]    = df.groupby("coin_id")["log_ret"].transform(lambda x: x.rolling(w).std())
    # Momentum difference: ret_ma_5 - ret_ma_50
    df["mom_diff_5_50"] = df["ret_ma_5"] - df["ret_ma_50"]
    # Additional simple moving averages
    df["sma_7"]        = df.groupby("coin_id")["close"].transform(lambda x: x.rolling(7).mean())
    df["sma_30"]       = df.groupby("coin_id")["close"].transform(lambda x: x.rolling(30).mean())
    # Trend ratio: sma_7 / sma_30
    df["trend_ratio"]  = df["sma_7"] / df["sma_30"]
    # Time-based cyclic features
    sec = df["timestamp"].values.astype("int64") // 10**9  # seconds since epoch
    day = 24 * 60 * 60
    week = 7 * day
    df["sin_day"]      = np.sin(2 * np.pi * sec / day)
    df["cos_day"]      = np.cos(2 * np.pi * sec / day)
    df["sin_week"]     = np.sin(2 * np.pi * sec / week)
    df["cos_week"]     = np.cos(2 * np.pi * sec / week)
    # List of feature columns
    features = [
        "volume_ratio","log_ret","range_pct","oc_pct",
        "sma_10","ema_10","ret_ma_5","vol_5",
        "sma_7","sma_30","trend_ratio",
        "sin_day","cos_day","sin_week","cos_week"
    ]
    # Drop rows with missing feature values
    return df.dropna(subset=features)

# -----------------------------------------------------------------------------
# 4) Main loop: train model, save signals & feature importances
# -----------------------------------------------------------------------------
# Create output directories if they do not exist
os.makedirs(RESULTS_ROOT, exist_ok=True)
os.makedirs(FEATURES_ROOT, exist_ok=True)

for iv in INTERVALS:
    print(f"Processing interval {iv}...")

    # Define file paths for train and test data under DATA_ROOT
    train_path = os.path.join(DATA_ROOT, f"train_{iv}.parquet")
    test_path  = os.path.join(DATA_ROOT, f"test_{iv}.parquet")

    # Load and preprocess datasets
    train_df = add_features(load_and_preprocess(train_path))
    test_df  = add_features(load_and_preprocess(test_path))

    # Identify feature columns (exclude id, timestamp, target)
    feature_cols = [c for c in train_df.columns if c not in ("coin_id","timestamp","target_direction")]

    # Set up time-series cross-validation and modeling pipeline
    tscv = TimeSeriesSplit(n_splits=N_SPLITS_CV)
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler",  StandardScaler()),
        ("clf",     LogisticRegression(class_weight="balanced", solver="liblinear", max_iter=1000))
    ])
    grid = GridSearchCV(pipe, param_grid=PARAM_GRID, cv=tscv, scoring="roc_auc", n_jobs=-1)

    # Create output folder for this interval's signals
    out_dir = os.path.join(RESULTS_ROOT, iv)
    os.makedirs(out_dir, exist_ok=True)

    all_importances = []
    # Iterate over each coin separately
    for coin in train_df["coin_id"].unique():
        tr = train_df[train_df["coin_id"] == coin]
        te = test_df[test_df["coin_id"] == coin]
        if tr.empty or te.empty:
            continue

        # Split into features (X) and target (y)
        X_tr, y_tr = tr[feature_cols], tr["target_direction"]
        X_te, y_te = te[feature_cols], te["target_direction"]

        # Fit model via grid search CV
        grid.fit(X_tr, y_tr)
        best = grid.best_estimator_

        # Extract feature importances (absolute coefficient values)
        coefs = best.named_steps["clf"].coef_[0]
        imp = pd.Series(np.abs(coefs), index=feature_cols).sort_values(ascending=False)
        df_imp = imp.reset_index()
        df_imp.columns = ["feature","importance"]
        df_imp["coin_id"] = coin
        df_imp["rank"]    = np.arange(1, len(df_imp) + 1)
        all_importances.append(df_imp)

        # Generate trading signals and compute returns
        df_te    = te.sort_values("timestamp").reset_index(drop=True)
        preds    = best.predict(df_te[feature_cols])
        signal   = pd.Series(preds, index=df_te["timestamp"]).shift(1).fillna(0).astype(int)
        bar_ret  = df_te.set_index("timestamp")["close"].pct_change().shift(-1).loc[signal.index]
        df_out   = pd.DataFrame({"signal": signal, "bar_ret": bar_ret}).dropna()
        # Save per-coin signal outputs
        df_out.to_parquet(os.path.join(out_dir, f"{coin}.parquet"))

    # Combine and save all feature importances for this interval
    df_all_imp = pd.concat(all_importances, ignore_index=True)
    csv_path   = os.path.join(FEATURES_ROOT, f"{iv}_feature_importance.csv")
    df_all_imp.to_csv(csv_path, index=False)
    print(f"  ▶ Feature importances saved to {csv_path}")

Processing interval 1min...


/opt/anaconda3/envs/erdos_summer_2025/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/opt/anaconda3/envs/erdos_summer_2025/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


  ▶ Feature importances saved to features/1min_feature_importance.csv
Processing interval 10min...


/opt/anaconda3/envs/erdos_summer_2025/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


  ▶ Feature importances saved to features/10min_feature_importance.csv
Processing interval 1h...


/opt/anaconda3/envs/erdos_summer_2025/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


  ▶ Feature importances saved to features/1h_feature_importance.csv
Processing interval 1d...
  ▶ Feature importances saved to features/1d_feature_importance.csv
